# **Project 4 - Spotify-Music-Trend-Analysis - ETL**

## Objectives

- Extract data from provided CSV file
- Clean it - check for duplicates/missing values if necessary negative numeric values
- Apply feature engineering if necessary 
- Remove any unnecessary columns
- Rename columns if necessary
- Save to a new CSV file


## Inputs

CSV files provided:

spotifydataset.csv


Note: original file stored in Data/OriginalFiles


## Outputs

CSV files created from ETL etc stored in Assets/Data/CleanedFiles and Assets/Data/VisualisationFiles:

spotifydataset_Cleaned.csv



## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"

Used AI tool used:
- GitHub 
 

See Documents/What_AI_Used_For.md for more details.


## Initalise Working Environment

In [1]:
#import libraries
import os
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [2]:
#DataFrame vars for ETL
dfSpotify_DataSet = None
dfSpotify_DataSet_Work = None
dfSpotify_DataSet_Temp = None
dfTemp = None
dfCleaned = None

#missing values check vars
intLessThanZero = 0

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
lstColumns = list()
intNum = 0
intMissingValues = 0

## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project3-Spotify/Spotify-Music-Trend-Analysis


## Create Project Folder Structure

In [4]:
#create project folder structure
modETL.funcCreateDirectories()

Folder Structure Created Successfully!


# Section 1 - Extraction

- Read csv file sales data-set.csv as other two do not require ETL (boo!)
- Move into working files directory
- Read csv file into a pandas dataframe
- Get schema info -> column and row numbers
- Get first 5 records
- Get list of column datatypes
- Get detailed schema infomation

## Read csv File Into Variable For Processing

In [6]:
#read csv file into DataFrame
dictDataFrames = modETL.funcReadOriginalFilesReturnDictionary()

dfSpotify_DataSet = dictDataFrames["spotifydataset.csv"]
#save as working csv file
modETL.funcSaveDataFrameToWorkingFile(dfSpotify_DataSet)

#read file from working files folder
#ETL library returns a dictionary of all files in the folder with the attribute name
#set to the actual csv filename
dictDataFrames = modETL.funcReadWorkingFilesReturnDictionary()
dfSpotify_DataSet = dictDataFrames["spotifydataset_Working.csv"]

dfSpotify_DataSet_Work = dfSpotify_DataSet.copy()

1 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset.csv


Saved: spotifydataset To Working Folder
1 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset_Working.csv




## Get Schema Info - Sales_Features_DataSet

In [7]:
modETL.funcGetStructure(dfSpotify_DataSet_Work)
dfSpotify_DataSet_Work.head()


DataFrame Structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0.1      114000 non-null  int64  
 1   Unnamed: 0        114000 non-null  int64  
 2   track_id          114000 non-null  object 
 3   artists           113999 non-null  object 
 4   album_name        113999 non-null  object 
 5   track_name        113999 non-null  object 
 6   popularity        114000 non-null  int64  
 7   duration_ms       114000 non-null  int64  
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16 

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Observations - Spotify_DataSet
114,000 rows
22 columns

Columns:

Unnamed0.1, Unamed0,track_id, artists, album_name, track_name, popularity, duration_ms, explicit, danceability, energy, key, loudness,
mode, speechiness, acousticness, instrumentalness, liveness, valence, temp, time_signature, track_genre

Assuming "Unnamed" are an index columns and can be dropped if necessary  

In the Summary of Dataframe Structure for numerical values we see this:

DataFrame Structure:  
===================  
<class 'pandas.DataFrame'>  
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):

|     |Column               |Non-Null Count |  Dtype  |    
|-----|---------------------|---------------|---------|  
 0   |Unnamed: 0.1      |114000 |int64 |
 1   |Unnamed: 0        |114000 |int64 |
 2   |track_id          |114000 |object| 
 3   |artists           |113999 |object| 
 4   |album_name        |113999 |object| 
 5   |track_name        |113999 |object| 
 6   |popularity        |114000 |int64 | 
 7   |duration_ms       |114000 |int64 | 
 8   |explicit          |114000 |bool  | 
 9   |danceability      |114000 |float64|
 10  |energy            |114000 |float64|
 11  |key               |114000 |int64  |
 12  |loudness          |114000 |float64|
 13  |mode              |114000 |int64  |
 14  |speechiness       |114000 |float64|
 15  |acousticness      |114000 |float64|
 16  |instrumentalness  |114000 |float64|
 17  |liveness          |114000 |float64|
 18  |valence           |114000 |float64|
 19  |tempo             |114000 |float64|
 20  |time_signature    |114000 |int64  |
 21  |track_genre       |114000 |object |

dtypes: bool(1), float64(9), int64(7), object(5)
memory usage: 18.4+ MB

Looks like there are some null values in the artists, album_name and track_name




# Get Column Data Types

In [56]:
#get column data types
print (f"Data Types: \n{dfSpotify_DataSet_Work.dtypes}")

Data Types: 
Unnamed: 0.1          int64
Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object


## Observations - Spotify_DataSet

No obvious candidates for transformation



## Get Schema Statistics

Look for duplicates and missing values

In [8]:
#get column statistics
modETL.funcGetStatistics(dfSpotify_DataSet_Work)

DataFrame Statistics:
        Unnamed: 0.1     Unnamed: 0     popularity   duration_ms  \
count  114000.000000  114000.000000  114000.000000  1.140000e+05   
mean    56999.500000   56999.500000      33.238535  2.280292e+05   
std     32909.109681   32909.109681      22.305078  1.072977e+05   
min         0.000000       0.000000       0.000000  0.000000e+00   
25%     28499.750000   28499.750000      17.000000  1.740660e+05   
50%     56999.500000   56999.500000      35.000000  2.129060e+05   
75%     85499.250000   85499.250000      50.000000  2.615060e+05   
max    113999.000000  113999.000000     100.000000  5.237295e+06   

        danceability         energy            key       loudness  \
count  114000.000000  114000.000000  114000.000000  114000.000000   
mean        0.566800       0.641383       5.309140      -8.258960   
std         0.173542       0.251529       3.559987       5.029337   
min         0.000000       0.000000       0.000000     -49.531000   
25%         0.456000

## Observations - Spotify_DataSet

Observations:
- Missing Values in artist, album_name and track_name - will need to see the record to determine if can remove
- High duplicates expected in shown columns as nature of data in them would create this situation



---

## Get Unique Values - Spotify_DataSet

In [9]:
#show unique values count
modETL.funcGetUniqueValuesCount(dfSpotify_DataSet_Work)

DataFrame Unique Values Per Column:
Unnamed: 0.1         - 114000: Unique Values Out Of 114000 Total Values
Unnamed: 0           - 114000: Unique Values Out Of 114000 Total Values
track_id             - 89741: Unique Values Out Of 114000 Total Values
artists              - 31437: Unique Values Out Of 113999 Total Values
album_name           - 46589: Unique Values Out Of 113999 Total Values
track_name           - 73608: Unique Values Out Of 113999 Total Values
popularity           - 101: Unique Values Out Of 114000 Total Values
duration_ms          - 50697: Unique Values Out Of 114000 Total Values
explicit             - 2: Unique Values Out Of 114000 Total Values
danceability         - 1174: Unique Values Out Of 114000 Total Values
energy               - 2083: Unique Values Out Of 114000 Total Values
key                  - 12: Unique Values Out Of 114000 Total Values
loudness             - 19480: Unique Values Out Of 114000 Total Values
mode                 - 2: Unique Values Out Of 114

## Observations - Spotify_DataSet

The columns where I would expect to see high unique values are:
- album_name
- track_name

This is shown in the results




## Get Categorical Value Distribution

In [ ]:
#get categorical distribution  
modETL.funcGetCategoricalValueDistribution(dfSpotify_DataSet_Work)


DataFrame Categorical Value Distribution:
[track_id] Value Distribution:
track_id
6S3JlDAGk3uu3NtZbPnuhS    9
2Ey6v4Sekh3Z0RUSISRosD    8
2kkvB3RNRzwjFdGhaUA0tz    8
5ZsAhuQ24mWHiduaxJqnhW    7
08kTa3SL9sV6Iy8KLKtGql    7
                         ..
0kJ7eKX6aWl8X1W5Xrosn6    1
4bYH5445Bn2w9UiGM0NxQw    1
1T5C6ENvpM3IiYeezsK9uI    1
34SatKRJgtXfL0bcgk7HMA    1
2hETkH7cOfqmz3LqZDHZf5    1
Name: count, Length: 89741, dtype: int64


[artists] Value Distribution:
artists
The Beatles                                                                 279
George Jones                                                                271
Stevie Wonder                                                               236
Linkin Park                                                                 224
Ella Fitzgerald                                                             222
                                                                           ... 
Automatic Tasty                                  

## Observations - Get Categorical Value Distribution

Some very interesting statistics here, expected to see common value for artists and genre, no obvious correalations

# Section 2

- If missing values determine what to fill with (mean, mode or categorical something else)
- Transform data types if necessary


## Strip Spaces From Object Columns

In [10]:
#remove spaces from all object columns (string datatype)

for col in dfSpotify_DataSet_Work.select_dtypes(include=['object']).columns:
    dfSpotify_DataSet_Work[col] = dfSpotify_DataSet_Work[col].str.strip()

          
#show first 50 rows to check    
dfSpotify_DataSet_Work.head(50)    

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
5,5,5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,...,-8.807,1,0.1050,0.2890,0.000000,0.1890,0.6660,98.017,4,acoustic
6,6,6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,False,0.407,...,-8.822,1,0.0355,0.8570,0.000003,0.0913,0.0765,141.284,3,acoustic
7,7,7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,...,-9.331,1,0.0417,0.5590,0.000000,0.0973,0.7120,150.960,4,acoustic
8,8,8,0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74,189613,False,0.625,...,-8.700,1,0.0369,0.2940,0.000000,0.1510,0.6690,130.088,4,acoustic
9,9,9,7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56,205594,False,0.442,...,-6.770,1,0.0295,0.4260,0.004190,0.0735,0.1960,78.899,4,acoustic


## Observations - Spotify_DataSet

No data corruptions

## Validate Numerical Values For Columns

see if any numerical column value is zero if so show count and column name


In [12]:
#see if any columns with int64 or float64 datatypes have values less than 1
for col in dfSpotify_DataSet_Work.select_dtypes(include=['int64', 'float64']).columns:
    intLessThanOne = (dfSpotify_DataSet_Work[col] < 0).sum()
    if intLessThanOne > 0:
       print(f"Number of Values In {col} Less Than 0: {intLessThanOne}")



Number of Values In loudness Less Than 0: 113910


# Show The Data

In [ ]:
# show negative loudness values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work[ dfSpotify_DataSet_Work["loudness"] <0 ]
dfSpotify_DataSet_Temp

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


---

## Observations - Spotify_DataSet

Not sure as to *why* there would be negative values in the loudness column, it is possible in audio, maybe this is to allow
playback software to know when to compensate and use compression during playback?

Will ask customer for more detail.

# Check Categorical Columns Have Values

We know:

artists
album_name
track_name

Have null values, but where are they?

In [32]:
#show missing values for artists column
dfSpotify_DataSet_Temp =  dfSpotify_DataSet_Work[dfSpotify_DataSet_Work["artists"].isnull()]
dfSpotify_DataSet_Temp
   


,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,65900,65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,...,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [33]:
#show missing values for album_name column
dfSpotify_DataSet_Temp =  dfSpotify_DataSet_Work[dfSpotify_DataSet_Work["album_name"].isnull()]
dfSpotify_DataSet_Temp

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,65900,65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,...,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [34]:
#show missing values for track_name column
dfSpotify_DataSet_Temp =  dfSpotify_DataSet_Work[dfSpotify_DataSet_Work["track_name"].isnull()]
dfSpotify_DataSet_Temp

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,65900,65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,...,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


# Observations Missing Values In Certain Columns

The missing values are all in the same row, being as there is only one row am happy to remove it from the data

# Delete Row With Missing Values

In [35]:
#delete rows with missing values in the artists, album_name, and track_name columns
dfSpotify_DataSet_Work = dfSpotify_DataSet_Work.dropna(subset=["artists", "album_name", "track_name"])

## Data Transformations/Feature Engineering

At this time can find nothing to do!

## Remove Unnecessary Columns

Will remove:

- Unnamed 0 column as it is an extra index column
- track_id has no use in analysis or visualisations



In [36]:
#going to remove:
dfSpotify_DataSet_Work.drop(columns=["Unnamed: 0.1","track_id"], inplace=True)

#check results
dfSpotify_DataSet_Work.info()

<class 'pandas.core.frame.DataFrame'>
Index: 113999 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        113999 non-null  int64  
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        113999 non-null  int64  
 5   duration_ms       113999 non-null  int64  
 6   explicit          113999 non-null  bool   
 7   danceability      113999 non-null  float64
 8   energy            113999 non-null  float64
 9   key               113999 non-null  int64  
 10  loudness          113999 non-null  float64
 11  mode              113999 non-null  int64  
 12  speechiness       113999 non-null  float64
 13  acousticness      113999 non-null  float64
 14  instrumentalness  113999 non-null  float64
 15  liveness          113999 non-null  float64
 16  valence           113999 

## Feature Engineering

At this time cannot think of anything to add!

## Observations - Feature Engineering



## Save Work DataFrame To CSV File

In [37]:
#create new DataFrame for cleaned data
dfCleaned = dfSpotify_DataSet_Work.copy()

#save the cleaned DataFrame to a new file
modETL.funcSaveDataFrameToCleanedFile(dfCleaned)



Saved: spotifydataset To Visualisation Folder


# Conclusions and Next Steps

ETL went well, DataSet ready for use